# Cardiac Patient Monitoring System
## 01 — Environment, Dataset Loading & Initial Cleaning (Milestone M1, Days 1–2)

**Project type:** Individual educational ML analysis (not a clinical system).

**Dataset:** UCI Heart Disease — Cleveland Database (ID 45), retrieved and validated in Phase 1.
303 observations, 13 features, target `num` (0–4), to be binarized to `target` (0 = absence, 1 = presence).

**This notebook covers:**
1. Environment / library check
2. Load raw dataset
3. Initial inspection (`head`, `tail`, `shape`, `info`, `describe`)
4. Missing value inspection
5. Duplicate inspection
6. Data type classification (numerical / categorical / target)
7. Save a lightly-cleaned, still-raw-preserving snapshot for Phase 2

**Rule:** the raw CSV in `data/raw/` is never modified. All cleaning output goes to `data/processed/`.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
RANDOM_STATE = 42

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.5.1
pandas: 3.0.3


## 1. Load the raw dataset

Source: UCI Machine Learning Repository — Heart Disease (Cleveland), DOI 10.24432/C52P4X.


In [2]:
RAW_PATH = "../data/raw/heart_disease_cleveland_raw.csv"

df = pd.read_csv(RAW_PATH)
df.shape


(303, 14)

In [3]:
df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [4]:
df.tail()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0,1
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0,2
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0,3
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0,1
302,38,1,3,138,175,0,0,173,0,0.0,1,NaN,3.0,0


In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB


In [6]:
df.describe(include='all').T


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## 2. Missing values

Per the data dictionary, missing values are expected only in `ca` (4) and `thal` (2).


In [7]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0')


,missing_count,missing_pct
ca,4,1.32
thal,2,0.66


**Interpretation:** Missing values are confined to `ca` (1.32%) and `thal` (0.66%) — a total of 6
cells out of 3,939 (303 × 13), i.e. ~0.15% of all feature values. This is small enough that a
simple, well-justified imputation strategy (applied later, inside a train-only-fit pipeline in
Phase 6) will not meaningfully bias the analysis. No manual imputation happens in this notebook —
that is deferred to the leakage-free pipeline stage.


## 3. Duplicate rows


In [8]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")


Duplicate rows: 0


## 4. Data type classification

Several columns are stored as numeric (int/float) but are conceptually **categorical** (encoded
categories, not continuous quantities). We classify them explicitly here; the actual dtype
conversion / encoding is handled later via `ColumnTransformer` in Phase 6 so we don't introduce
leakage or lock in a representation too early.


In [9]:
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
target_col_raw = 'num'

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Raw target:", target_col_raw)

assert set(numerical_features) | set(categorical_features) | {target_col_raw} == set(df.columns)


Numerical features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Raw target: num


## 5. Target inspection (raw, 5-class)


In [10]:
df['num'].value_counts().sort_index()


num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

## 6. Binary target definition (documented transform, not yet applied to raw file)

Per the data dictionary (§7): `num == 0` → absence (0); `num` in {1,2,3,4} → presence (1).
This transform will be applied to a **derived** column, never overwriting `num`, and only inside
the processed dataset created below.


In [11]:
df_processed = df.copy()
df_processed['target'] = (df_processed['num'] > 0).astype(int)

df_processed['target'].value_counts().rename({0: 'absence (0)', 1: 'presence (1)'})


target
absence (0)     164
presence (1)    139
Name: count, dtype: int64

In [12]:
df_processed['target'].value_counts(normalize=True).round(3)


target
0    0.541
1    0.459
Name: proportion, dtype: float64

**Interpretation:** ~54% absence vs ~46% presence — close to balanced. No aggressive
resampling (e.g. SMOTE) is needed; stratified splitting and stratified k-fold CV will be
sufficient to keep class proportions consistent across train/test/CV folds.


## 7. Save processed snapshot for Phase 2

This file still contains the raw `num` column (untouched) plus the new `target` column.
No imputation, scaling, or encoding has been applied yet — that happens in later phases,
fit only on training data.


In [13]:
import os
os.makedirs("../data/processed", exist_ok=True)
df_processed.to_csv("../data/processed/heart_disease_cleveland_stage1.csv", index=False)
print("Saved:", os.path.abspath("../data/processed/heart_disease_cleveland_stage1.csv"))


Saved: c:\Users\HP\Desktop\BinX_ML_Internship\Cardiac_Patient_Monitoring_System_Project\data\processed\heart_disease_cleveland_stage1.csv


## M1 Quality Gate — Checklist

- [x] Environment loads (numpy, pandas, matplotlib)
- [x] Dataset loads successfully from `data/raw/`
- [x] Dataset inspected (`head`, `tail`, `info`, `describe`)
- [x] Missing values identified and quantified (`ca`: 4, `thal`: 2)
- [x] Duplicates checked
- [x] Columns classified into numerical / categorical / target
- [x] Binary target defined and documented (raw `num` preserved)
- [x] Processed stage-1 snapshot saved to `data/processed/`

**Next:** Phase 2 — Data Cleaning + Data Quality (missing-value strategy, invalid-value checks,
categorical encoding plan, data-quality report).
